# Assign Classified Bakery establishments to London MSOAs

This notebook assigns the classified Bakery establishments that we find in the first part of the project to the 2021 MSOA with postcodes as the main method and longitude and latitude used as a backup in case postcodes are missing.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd


DATA_DATE = "2026-07-23"

BAKERY_PATH = Path(f"../data/business/processed/london_bakery_establishments_final_{DATA_DATE}.csv")
MSOA_PATH = Path("../data/spatial/processed/london_msoa_2021.gpkg")
POSTCODE_LOOKUP_PATH = Path("../data/spatial/raw/msoa_lookup_data.csv")

PROCESSED_SPATIAL_FOLDER = Path("../data/spatial/processed")
PROCESSED_SPATIAL_FOLDER.mkdir(parents=True, exist_ok=True)

BAKERY_MSOA_OUTPUT_PATH = (PROCESSED_SPATIAL_FOLDER / f"london_bakery_establishments_with_msoa_{DATA_DATE}.csv")
MSOA_COUNTS_OUTPUT_PATH = (PROCESSED_SPATIAL_FOLDER / f"london_msoa_bakery_counts_{DATA_DATE}.csv")
UNMATCHED_OUTPUT_PATH = (PROCESSED_SPATIAL_FOLDER / f"london_bakery_establishments_unmatched_msoa_{DATA_DATE}.csv")

EXPECTED_BAKERIES = 5221
EXPECTED_MSOAS = 1002

# Checking column names and data structure

Since we will be joining these two datasets, it is important to keep note of the columns of data available and the names that will be required in the final join.

In [2]:
bakeries = pd.read_csv(BAKERY_PATH, low_memory= False)
london_msoa = gpd.read_file(MSOA_PATH)
postcode_lookup = pd.read_csv(POSTCODE_LOOKUP_PATH, usecols=["pcds", "msoa21cd"], dtype="string")

bakeries["FHRSID"] = (pd.to_numeric(bakeries["FHRSID"], errors="coerce")
                      .astype("Int64"))


print(f"Bakery establishments: {len(bakeries)}")
print(f"London MSOA rows: {len(london_msoa)}")
print(f"Postcode lookup rows: {len(postcode_lookup)}")

print(f"\nBakery columns: {bakeries.columns.tolist()}")
print(f"\nMSOA columns: {london_msoa.columns.tolist()}")

Bakery establishments: 5221
London MSOA rows: 1002
Postcode lookup rows: 2726477

Bakery columns: ['FHRSID', 'BusinessName', 'BusinessNameClean', 'BusinessType', 'Address', 'PostCode', 'LocalAuthorityName', 'geocode.longitude', 'geocode.latitude', 'FinalClass', 'ClassificationSource']

MSOA columns: ['borough_name', 'borough_code', 'msoa_name', 'msoa_code', 'area_km2', 'geometry']


## Prepare postcodes and coordinates

Since MSOA's will be assigned by joining on postcodes and geocodes, these need to be formatted the same way.

In [3]:
bakeries["postcode_clean"] = (bakeries["PostCode"]
                              .astype("string")
                              .str.upper()
                              .str.replace(" ", "", regex=False)
                              .str.strip())

postcode_lookup["postcode_clean"] = (postcode_lookup["pcds"]
                                     .astype("string")
                                     .str.upper()
                                     .str.replace(" ", "", regex=False)
                                     .str.strip())

bakeries["geocode.longitude"] = pd.to_numeric(bakeries["geocode.longitude"],
                                              errors="coerce")

bakeries["geocode.latitude"] = pd.to_numeric(bakeries["geocode.latitude"],
                                             errors="coerce")

missing_postcode = (bakeries["postcode_clean"].isna() | bakeries["postcode_clean"].eq(""))
missing_coordinates = (bakeries["geocode.longitude"].isna() | bakeries["geocode.latitude"].isna())

missing_both = missing_postcode & missing_coordinates

print(f"Bakery establishments: {len(bakeries)}")
print(f"Missing postcode: {missing_postcode.sum()}")
print(f"Missing complete coordinates: {missing_coordinates.sum()}")
print(f"Missing both postcode and coordinates: {missing_both.sum()}")

Bakery establishments: 5221
Missing postcode: 35
Missing complete coordinates: 282
Missing both postcode and coordinates: 0


In [4]:
london_msoa_codes = set(london_msoa["msoa_code"])

postcode_lookup_london = (postcode_lookup[postcode_lookup["msoa21cd"]
                                          .isin(london_msoa_codes)
                                          ][["postcode_clean", "msoa21cd"]]
                                          .dropna(subset=["postcode_clean"])
                                          .drop_duplicates(subset="postcode_clean")
                                          .rename(columns={"msoa21cd": "msoa_code"}))

bakeries_with_msoa = bakeries.merge(postcode_lookup_london,
                                    on="postcode_clean",
                                    how="left",
                                    validate="many_to_one")

bakeries_with_msoa["assignment_method"] = "unmatched"

postcode_matched = bakeries_with_msoa["msoa_code"].notna()

bakeries_with_msoa.loc[postcode_matched, "assignment_method"
                       ] = "postcode"

print(f"Matched using postcode: {postcode_matched.sum()}")
print(f"Still unmatched: {(~postcode_matched).sum()}")

Matched using postcode: 5143
Still unmatched: 78


# Assigning unmatched establishments using coordinates

Using EPSG:4326 we can convert the spatial points from the geocode to match the London MSOA CRS. The join will assign each point to the MSOA polygon that it is a part of and the successful joins are then added back into the msoa_code column and those that are left unmatched will be assigned as such for future reference.

In [5]:
coordinate_candidates = bakeries_with_msoa[bakeries_with_msoa["msoa_code"].isna()
                                           & bakeries_with_msoa["geocode.longitude"].notna()
                                           & bakeries_with_msoa["geocode.latitude"].notna()].copy()

coordinate_match_count = 0

print(f"Unmatched establishments with usable coordinates: {len(coordinate_candidates)}")

if len(coordinate_candidates) > 0:
    coordinate_points = gpd.GeoDataFrame(coordinate_candidates,
                                         geometry=gpd.points_from_xy(coordinate_candidates["geocode.longitude"],
                                                                     coordinate_candidates["geocode.latitude"]),
                                                                     crs="EPSG:4326").to_crs(london_msoa.crs)

    msoa_for_join = (london_msoa[["msoa_code", "geometry"]]
                     .rename(columns={"msoa_code": "coordinate_msoa_code"}))

    coordinate_matches = gpd.sjoin(coordinate_points, msoa_for_join,
                                   how="left",
                                   predicate="within")

    coordinate_matches = coordinate_matches.dropna(subset=["coordinate_msoa_code"])

    coordinate_match_count = len(coordinate_matches)

    bakeries_with_msoa.loc[coordinate_matches.index, "msoa_code"
                           ] = coordinate_matches["coordinate_msoa_code"]

    bakeries_with_msoa.loc[coordinate_matches.index, "assignment_method"
                           ] = "coordinates"

print(f"Matched using coordinates: {coordinate_match_count}")

Unmatched establishments with usable coordinates: 36
Matched using coordinates: 35


# Validation on MSOA lookup of Bakery establishments

A final validation and check is done to understand how successful this method was in assigning bakery establishments to their MSOA codes. Those without MSOA codes assigned may have to be removed or manually inspected and assigned.

In [6]:
final_matched = bakeries_with_msoa["msoa_code"].notna().sum()
final_unmatched = bakeries_with_msoa["msoa_code"].isna().sum()
match_rate = final_matched / len(bakeries_with_msoa) * 100

print("Assignment methods:")
display(bakeries_with_msoa["assignment_method"]
        .value_counts()
        .rename("Establishments"))

print(f"Original bakery rows: {len(bakeries)}")
print(f"Final bakery rows: {len(bakeries_with_msoa)}")
print(f"Assigned to an MSOA: {final_matched}")
print(f"Still unmatched: {final_unmatched}")
print(f"Match rate: {match_rate:.2f}%")

if final_unmatched > 0:
    display(bakeries_with_msoa.loc[bakeries_with_msoa["msoa_code"].isna(),
                                   ["FHRSID",
                                    "BusinessName",
                                    "Address",
                                    "PostCode",
                                    "LocalAuthorityName",
                                    "geocode.longitude",
                                    "geocode.latitude"]])

Assignment methods:


assignment_method
postcode       5143
unmatched        43
coordinates      35
Name: Establishments, dtype: int64

Original bakery rows: 5221
Final bakery rows: 5221
Assigned to an MSOA: 5178
Still unmatched: 43
Match rate: 99.18%


,FHRSID,BusinessName,Address,PostCode,LocalAuthorityName,geocode.longitude,geocode.latitude
22,1752355,Sugaroholic,NaN,RM10,Barking and Dagenham,NaN,NaN
47,1205496,Illegally Delicious,NaN,RM8,Barking and Dagenham,NaN,NaN
125,1960730,Minique Atelier,NaN,N12,Barnet,NaN,NaN
165,1407329,CJ's Bakery,NaN,N3 2,Barnet,NaN,NaN
233,1914232,Kaaphi,NaN,NW7,Barnet,NaN,NaN
347,1765006,Sourdough Mikate,NaN,SE2,Bexley,NaN,NaN
492,1753666,Freads Breads,NaN,NW6,Brent,NaN,NaN
506,1921704,Momo Patisserie,NaN,NW9,Brent,NaN,NaN
556,1842653,Dooley Dough,NaN,BR2,Bromley,NaN,NaN
570,940534,Elvira's Secret Pantry,NaN,BR3,Bromley,NaN,NaN


## Create MSOA bakery counts

MSOA names and borough information are now attached to the bakery establishments, which can be counted within each MSOA. The count is joined to the complete set of 1,002 London MSOAs so that areas containing no identified bakeries are retained with a count of zero.

In [7]:
msoa_details = london_msoa[["msoa_code",
                            "msoa_name",
                            "borough_code",
                            "borough_name"]].copy()

bakeries_with_msoa = bakeries_with_msoa.merge(msoa_details,
                                              on="msoa_code",
                                              how="left",
                                              validate="many_to_one")

bakery_counts = (bakeries_with_msoa
                 .dropna(subset=["msoa_code"])
                 .groupby("msoa_code")
                 .size()
                 .rename("bakery_count")
                 .reset_index())

msoa_bakery_counts = msoa_details.merge(bakery_counts,
                                        on="msoa_code",
                                        how="left",
                                        validate="one_to_one")

msoa_bakery_counts["bakery_count"] = (msoa_bakery_counts["bakery_count"]
                                      .fillna(0)
                                      .astype(int))

print(f"MSOA rows: {len(msoa_bakery_counts)}")
print(f"Total bakeries counted: {msoa_bakery_counts['bakery_count'].sum()}")
print(f"MSOAs with no bakeries: {(msoa_bakery_counts['bakery_count'] == 0).sum()}")

display(msoa_bakery_counts["bakery_count"].describe())

MSOA rows: 1002
Total bakeries counted: 5178
MSOAs with no bakeries: 120


count    1002.000000
mean        5.167665
std         7.898193
min         0.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       129.000000
Name: bakery_count, dtype: float64

In [10]:
matched_bakery_count = (bakeries_with_msoa["msoa_code"]
                        .notna()
                        .sum())

unmatched_bakeries = (bakeries_with_msoa[bakeries_with_msoa["msoa_code"]
                                         .isna()].copy())


if len(bakeries_with_msoa) != EXPECTED_BAKERIES:
    raise RuntimeError(f"Expected {EXPECTED_BAKERIES} bakery rows, but found {len(bakeries_with_msoa)}.")

if bakeries_with_msoa["FHRSID"].duplicated().any():
    raise RuntimeError("Duplicate FHRSID values found after MSOA assignment.")

if len(msoa_bakery_counts) != EXPECTED_MSOAS:
    raise RuntimeError(f"Expected {EXPECTED_MSOAS} MSOA rows, but found {len(msoa_bakery_counts)}.")

if msoa_bakery_counts["bakery_count"].sum() != matched_bakery_count:
    raise RuntimeError("The MSOA bakery counts do not sum to the final bakery total.")

bakeries_with_msoa = (bakeries_with_msoa
                      .sort_values(["msoa_code", "BusinessName", "FHRSID"])
                      .reset_index(drop=True))

msoa_bakery_counts = (msoa_bakery_counts
                      .sort_values("msoa_code")
                      .reset_index(drop=True))

unmatched_bakeries = (unmatched_bakeries
                      .sort_values(["LocalAuthorityName", "BusinessName"])
                      .reset_index(drop=True))

bakeries_with_msoa.to_csv(BAKERY_MSOA_OUTPUT_PATH, index=False, encoding="utf-8-sig")
msoa_bakery_counts.to_csv(MSOA_COUNTS_OUTPUT_PATH, index=False, encoding="utf-8-sig")
unmatched_bakeries.to_csv(UNMATCHED_OUTPUT_PATH, index=False, encoding="utf-8-sig")

unmatched_percentage = (len(unmatched_bakeries) / len(bakeries_with_msoa)* 100)

print("Final validation passed.")


print(f"\nTotal bakery establishments: {len(bakeries_with_msoa)}")
print(f"Assigned to an MSOA: {matched_bakery_count}")
print(f"Excluded from MSOA counts: {len(unmatched_bakeries)} ({unmatched_percentage:.2f}%)")
print(f"Total bakeries in MSOA count dataset: {msoa_bakery_counts["bakery_count"].sum()}")

print(f"\nBakery establishment dataset saved to: {BAKERY_MSOA_OUTPUT_PATH}")
print(f"MSOA bakery count dataset saved to: {MSOA_COUNTS_OUTPUT_PATH}")


Final validation passed.

Total bakery establishments: 5221
Assigned to an MSOA: 5178
Excluded from MSOA counts: 43 (0.82%)
Total bakeries in MSOA count dataset: 5178

Bakery establishment dataset saved to: ..\data\spatial\processed\london_bakery_establishments_with_msoa_2026-07-23.csv
MSOA bakery count dataset saved to: ..\data\spatial\processed\london_msoa_bakery_counts_2026-07-23.csv
